# 05 - XGBoost challenger

Same development loans and split as the scorecard. I use the raw origination fields this time, without WoE. The five folds use training loans only, with a new category encoder in each fold.

I try a small set of tree depths and class weighting choices, then check PD calibration on reserved validation loans. The test split and new 2006 Q1 loans are left for evaluation.


In [3]:
# Same loans and split as logistic; raw origination fields for the tree model.
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from cohort import NUMERIC, features_from

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
OUT = DATA / "pocket1_model"
Y07, Y08, SEED = "default_180_plus_2007", "default_180_plus_2008", 42
RAW_DEV = DATA / "model_2005q1_dec2006_2007.parquet"
USE_MONOTONIC = True  # FICO down; CLTV, LTV and DTI up.
MIN_BRIER_GAIN = 1e-5

pipe = joblib.load(OUT / "feature_pipeline.joblib")
FEATURES = pipe["features"]
NUM = [f for f in NUMERIC if f in FEATURES]
CAT = [f for f in FEATURES if f not in NUM]
MONO = {"fico": -1, "cltv": 1, "ltv": 1, "dti": 1} if USE_MONOTONIC else {}
raw = pd.read_parquet(RAW_DEV)
ids = pd.read_parquet(OUT / "loan_splits.parquet")
oot = pd.read_parquet(OUT / "oot_2006q1_dec2007_2008_raw.parquet")
log_dev = pd.read_parquet(OUT / "scored_loans.parquet")
log_oot = pd.read_parquet(OUT / "oot_2006q1_dec2007_2008_scored.parquet")
if set(raw["loan_id"]) & set(oot["loan_id"]):
    raise ValueError("Development and 2008 test share loan IDs")
dev = raw.merge(ids[["loan_id", "split"]], on="loan_id", how="inner", validate="one_to_one")
if len(dev) != len(raw):
    raise ValueError("Some development loans have no saved split")
splits = {k: dev.loc[dev["split"].eq(k)].reset_index(drop=True) for k in
          ["train", "validation", "test"]}
print("Split sizes:", {k: len(v) for k, v in splits.items()}, "| 2008 OOT:", len(oot))

Split sizes: {'train': 171712, 'validation': 57238, 'test': 57238} | 2008 OOT: 253343


In [4]:
def new_encoder():
    # Fresh categories for each CV fold.
    return ColumnTransformer([
        ("num", "passthrough", NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                              dtype=np.float32), CAT)
    ], verbose_feature_names_out=True).set_output(transform="pandas")

def new_xgb(y_fold, max_depth=3, use_weight=False, early_stopping_rounds=None):
    pos, neg = int(np.sum(y_fold)), len(y_fold) - int(np.sum(y_fold))
    return XGBClassifier(n_estimators=500 if early_stopping_rounds else 200,
                         max_depth=max_depth, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         objective="binary:logistic", eval_metric="logloss",
                         tree_method="hist", random_state=SEED, n_jobs=4,
                         early_stopping_rounds=early_stopping_rounds,
                         scale_pos_weight=(neg / pos if use_weight else 1.0),
                         monotone_constraints={f"num__{f}": direction for f, direction in MONO.items()
                                               if f in NUM})

def measure(y, pd_hat, model_name, split):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(pd_hat, dtype=float), 1e-9, 1-1e-9)
    auc = roc_auc_score(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    return dict(split=split, model=model_name, loans=len(y), defaults=int(y.sum()),
                AUC=auc, PR_AUC=average_precision_score(y, p),
                KS=float((tpr-fpr).max()), Gini=2*auc-1,
                Brier=brier_score_loss(y, p), log_loss=log_loss(y, p, labels=[0, 1]),
                observed=y.mean(), predicted=p.mean())

X = {name: features_from(loans, FEATURES) for name, loans in splits.items()}
y_train = splits["train"][Y07].astype(int).to_numpy()

In [5]:
# Five folds inside the train split.
cv_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (fit_idx, hold_idx) in enumerate(cv.split(X["train"], y_train), start=1):
    enc = new_encoder()
    x_fit = enc.fit_transform(X["train"].iloc[fit_idx])
    x_hold = enc.transform(X["train"].iloc[hold_idx])
    m = new_xgb(y_train[fit_idx])
    m.fit(x_fit, y_train[fit_idx])
    p = m.predict_proba(x_hold)[:, 1]
    cv_rows.append({"fold": fold, **measure(y_train[hold_idx], p, "XGBoost", "CV holdout")})
cv_results = pd.DataFrame(cv_rows)
print("Five training folds:")
print(cv_results[["fold", "AUC", "KS", "Brier", "log_loss"]].round(5).to_string(index=False))
print("Mean +/- SD:\n", cv_results[["AUC", "KS", "Brier", "log_loss"]].agg(["mean", "std"]).round(5))

Five training folds:
 fold     AUC      KS   Brier  log_loss
    1 0.79200 0.50105 0.00197   0.01329
    2 0.80650 0.51929 0.00198   0.01314
    3 0.78548 0.51279 0.00195   0.01329
    4 0.79352 0.46857 0.00194   0.01305
    5 0.83505 0.58933 0.00197   0.01270
Mean +/- SD:
           AUC       KS    Brier  log_loss
mean  0.80251  0.51821  0.00196   0.01309
std   0.01972  0.04429  0.00001   0.00024


In [6]:
# Two depths, two weighting choices. Split validation for tuning and calibration.
y_val = splits["validation"][Y07].astype(int).to_numpy()
tune_idx, cal_idx = train_test_split(np.arange(len(y_val)), test_size=.50,
                                     stratify=y_val, random_state=SEED)
encoder = new_encoder()
x_train = encoder.fit_transform(X["train"])
x_tune = encoder.transform(X["validation"].iloc[tune_idx])
trials = []
for depth in (2, 3):
    for weighted in (False, True):
        candidate = new_xgb(y_train, max_depth=depth, use_weight=weighted,
                            early_stopping_rounds=30)
        candidate.fit(x_train, y_train, eval_set=[(x_tune, y_val[tune_idx])], verbose=False)
        pred = candidate.predict_proba(x_tune)[:, 1]
        trials.append(dict(depth=depth, class_weight=weighted,
                           best_iteration=candidate.best_iteration,
                           AUC=roc_auc_score(y_val[tune_idx], pred),
                           Brier=brier_score_loss(y_val[tune_idx], pred), model=candidate))
trial_report = pd.DataFrame([{k:v for k,v in row.items() if k != "model"} for row in trials])
best_candidate = min(trials, key=lambda row: (row["Brier"], -row["AUC"]))
xgb = best_candidate["model"]
print(trial_report.round(5).to_string(index=False))
print("Selected depth / weighted / iteration:", best_candidate["depth"],
      best_candidate["class_weight"], best_candidate["best_iteration"])

def raw_pd(x):
    return xgb.predict_proba(encoder.transform(x))[:, 1]

raw_validation = raw_pd(X["validation"])


 depth  class_weight  best_iteration     AUC   Brier
     2         False             225 0.81687 0.00195
     2          True             395 0.81304 0.16366
     3         False             161 0.81837 0.00195
     3          True             497 0.80139 0.13243
Selected depth / weighted / iteration: 2 False 225


In [7]:
# Check raw PDs against sigmoid and isotonic calibration.
def logit_of(p):
    z = np.clip(np.asarray(p, dtype=float), 1e-8, 1-1e-8)
    return np.log(z / (1-z)).reshape(-1, 1)

def fit_calibrator(method, probs, labels):
    labels = np.asarray(labels, dtype=int)
    if method == "sigmoid":
        cal = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1500)
        cal.fit(logit_of(probs), labels)
        return cal
    if method == "isotonic":
        return IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(probs, labels)
    return None

def apply_calibrator(probs, method, cal):
    if method == "sigmoid":
        return cal.predict_proba(logit_of(probs))[:, 1]
    if method == "isotonic":
        return cal.predict(np.asarray(probs, dtype=float))
    return np.asarray(probs, dtype=float)

cal_fit_idx, cal_check_idx = train_test_split(cal_idx, test_size=.50,
                                              stratify=y_val[cal_idx], random_state=SEED)
choices = [{"method": "none", "check_Brier": brier_score_loss(y_val[cal_check_idx],
                                                      raw_validation[cal_check_idx])}]
for method in ("sigmoid", "isotonic"):
    if min(np.sum(y_val[cal_fit_idx]), np.sum(1-y_val[cal_fit_idx])) < 25:
        continue  # Too few defaults here to trust the calibration fit.
    cal = fit_calibrator(method, raw_validation[cal_fit_idx], y_val[cal_fit_idx])
    p_check = apply_calibrator(raw_validation[cal_check_idx], method, cal)
    if method == "sigmoid" and cal.coef_[0, 0] <= 0:
        continue  # This would reverse the risk ordering.
    choices.append({"method": method, "check_Brier": brier_score_loss(y_val[cal_check_idx], p_check)})
cal_choices = pd.DataFrame(choices).sort_values("check_Brier")
raw_error = cal_choices.set_index("method").loc["none", "check_Brier"]
best = cal_choices.iloc[0]
cal_method = best["method"] if raw_error - best["check_Brier"] >= MIN_BRIER_GAIN else "none"
calibrator = fit_calibrator(cal_method, raw_validation[cal_idx], y_val[cal_idx]) if cal_method != "none" else None
print("Held-out validation calibration comparison:\n", cal_choices.round(6).to_string(index=False))
print("Calibration:", cal_method)

def predict_pd(x):
    return np.clip(apply_calibrator(raw_pd(x), cal_method, calibrator), 1e-9, 1-1e-9)

Held-out validation calibration comparison:
   method  check_Brier
 sigmoid     0.002014
    none     0.002015
isotonic     0.002028
Calibration: none


In [8]:
# Join by loan ID before comparing the PDs.
def attach_logistic(df, logistic, split_name=None):
    old = logistic if split_name is None else logistic.loc[logistic["split"].eq(split_name)]
    both = df.merge(old[["loan_id", "PD"]].rename(columns={"PD": "logistic_PD"}),
                    on="loan_id", how="left", validate="one_to_one", indicator=True)
    if (both["_merge"] != "both").any() or len(both) != len(old):
        raise ValueError(f"Champion/challenger populations mismatch: {split_name}")
    return both.drop(columns="_merge")

rows, dev_preds = [], []
for name in ("validation", "test"):
    d = splits[name]
    both = d[["loan_id", Y07]].copy()
    both["xgb_raw_PD"] = raw_pd(X[name])
    both["xgb_PD"] = predict_pd(X[name])
    both = attach_logistic(both, log_dev, name)
    rows.extend([measure(both[Y07], both["xgb_PD"], "XGBoost", name),
                 measure(both[Y07], both["logistic_PD"], "Logistic", name)])
    dev_preds.append(both.assign(split=name))
dev_results = pd.DataFrame(rows)
dev_predictions = pd.concat(dev_preds, ignore_index=True)
print(dev_results.round(5).to_string(index=False))

     split    model  loans  defaults     AUC  PR_AUC      KS    Gini   Brier  log_loss  observed  predicted
validation  XGBoost  57238       113 0.83335 0.00944 0.54878 0.66669 0.00196   0.01273   0.00197    0.00199
validation Logistic  57238       113 0.82361 0.00775 0.52576 0.64722 0.00196   0.01290   0.00197    0.00201
      test  XGBoost  57238       112 0.79253 0.00997 0.43410 0.58506 0.00195   0.01295   0.00196    0.00198
      test Logistic  57238       112 0.77899 0.00749 0.42986 0.55798 0.00195   0.01311   0.00196    0.00200


In [9]:
# Score 2008 and report results for resolved outcomes.
X_oot = features_from(oot, FEATURES)
oot_predictions = oot[["loan_id", Y08]].copy()
oot_predictions["xgb_raw_PD"] = raw_pd(X_oot)
oot_predictions["xgb_PD"] = predict_pd(X_oot)
oot_predictions = attach_logistic(oot_predictions, log_oot)
known = oot_predictions.loc[oot_predictions[Y08].notna()].copy()
oot_results = pd.DataFrame([
    measure(known[Y08], known["xgb_PD"], "XGBoost", "2008 OOT"),
    measure(known[Y08], known["logistic_PD"], "Logistic", "2008 OOT")])
known["decile"] = pd.qcut(-known["xgb_PD"], 10, labels=False, duplicates="drop") + 1
calibration = known.groupby("decile").agg(loans=(Y08, "size"),
    defaults=(Y08, "sum"), observed_PD=(Y08, "mean"), predicted_PD=("xgb_PD", "mean"))
print("2008 OOT: ", len(known), "verified;", len(oot_predictions)-len(known), "unresolved")
print(oot_results.round(5).to_string(index=False))
print("XGBoost OOT calibration by decile:\n", calibration.round(5))

2008 OOT:  253094 verified; 249 unresolved
   split    model  loans  defaults     AUC  PR_AUC      KS    Gini   Brier  log_loss  observed  predicted
2008 OOT  XGBoost 253094      1975 0.76766 0.02250 0.40382 0.53532 0.00772   0.04461    0.0078    0.00325
2008 OOT Logistic 253094      1975 0.76235 0.02105 0.39174 0.52469 0.00772   0.04447    0.0078    0.00328
XGBoost OOT calibration by decile:
         loans  defaults  observed_PD  predicted_PD
decile                                            
1       25310       630      0.02489       0.01476
2       25309       433      0.01711       0.00645
3       25309       282      0.01114       0.00403
4       25310       225      0.00889       0.00265
5       25309       181      0.00715       0.00173
6       25309       102      0.00403       0.00116
7       25310        74      0.00292       0.00078
8       25309        31      0.00122       0.00051
9       25309        10       0.0004       0.00030
10      25310         7      0.00028      

In [10]:
# Quick directional sweep: change one numeric field at a time.
scenarios = []
reference = X["train"].sample(n=min(800, len(X["train"])), random_state=SEED)
for f, direction in MONO.items():
    if f not in NUM or reference[f].dropna().nunique() < 3:
        continue
    grid = np.unique(np.quantile(reference[f].dropna(), np.linspace(.10, .90, 8)))
    means = []
    for value in grid:
        case = reference.copy()
        case[f] = float(value)
        means.append(float(np.mean(raw_pd(case))))
        scenarios.append({"feature": f, "value": value, "mean_raw_PD": means[-1]})
    change = np.diff(means)
    violations = int(np.sum(change * direction < -1e-8))
    print(f, "direction", direction, "violations", violations)
monotonic_audit = pd.DataFrame(scenarios)

fico direction -1 violations 0
cltv direction 1 violations 0
ltv direction 1 violations 0
dti direction 1 violations 0


In [11]:
# Keep loan-level PDs for the final review.
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(dict(model=xgb, encoder=encoder, calibrator=calibrator,
                 calibration_method=cal_method, features=FEATURES, numeric=NUM,
                 categorical=CAT, monotonic_constraints=MONO,
                 use_class_weight=best_candidate["class_weight"], development_target=Y07,
                 oot_target=Y08), OUT / "xgb_challenger.joblib")
cv_results.to_csv(OUT / "xgb_cv_5fold.csv", index=False)
trial_report.to_csv(OUT / "xgb_candidate_comparison.csv", index=False)
cal_choices.to_csv(OUT / "xgb_calibration_selection.csv", index=False)
dev_results.to_csv(OUT / "xgb_development_comparison.csv", index=False)
oot_results.to_csv(OUT / "xgb_2008_comparison.csv", index=False)
calibration.to_csv(OUT / "xgb_2008_calibration.csv")
monotonic_audit.to_csv(OUT / "xgb_monotonic_audit.csv", index=False)
dev_predictions.to_parquet(OUT / "xgb_development_predictions.parquet", index=False)
oot_predictions.to_parquet(OUT / "xgb_2008_predictions.parquet", index=False)
print("Saved Notebook 5 outputs to", OUT)

Saved Notebook 5 outputs to E:\CRM Projects\Data\pocket1_model
